# Loads environment variables

In [26]:
from dotenv import load_dotenv

load_dotenv()

python-dotenv could not parse statement starting at line 3


True

# Loads pdf.

In [27]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "Food_Recipes.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()

print(f"Loaded {len(docs)} pages")

for i, doc in enumerate(docs):
    print(f"\n{'=' * 60}")
    print(f"PAGE {i + 1}")
    print(f"{'=' * 60}")

    print("Content:")
    print(doc.page_content)

    print("\nMetadata:")
    print(doc.metadata)

Loaded 202 pages

PAGE 1
Content:
South & North Indian Recipes  |  Page 1
 SOUTH & NORTH INDIAN RECIPES
 A 200-page practical cookbook of traditional and everyday Indian dishes
 Recipe Collection
 
South Indian  North Indian  Regional Specialities  Snacks  Sweets  Cooking Tips

Metadata:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-22T08:57:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-22T08:57:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Food_Recipes.pdf', 'total_pages': 202, 'page': 0, 'page_label': '1'}

PAGE 2
Content:
South & North Indian Recipes  |  Page 2
How to Use This Book
Each recipe includes a simple ingredient list, preparation method, serving guidance and a practical cooking note.
Quantities can be adjusted according to family size and taste.

Metadata:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)',

# Splits the PDF into chunks.

In [28]:
from langchain_text_splitters import CharacterTextSplitter
 
# Split on newlines with fixed size
splitter = CharacterTextSplitter(
    separator="\n",        # Split at line breaks
    chunk_size=500,        # Max characters per chunk
    chunk_overlap=50,      # Small overlap
    length_function=len
)
 
chunks = splitter.split_documents(docs)
print(chunks)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-22T08:57:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-22T08:57:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Food_Recipes.pdf', 'total_pages': 202, 'page': 0, 'page_label': '1'}, page_content='South & North Indian Recipes  |  Page 1\n SOUTH & NORTH INDIAN RECIPES\n A 200-page practical cookbook of traditional and everyday Indian dishes\n Recipe Collection\n \nSouth Indian \x7f North Indian \x7f Regional Specialities \x7f Snacks \x7f Sweets \x7f Cooking Tips'), Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-22T08:57:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-22T08:57:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Food_Recipes.pdf', 'total

# Creates OpenAI embeddings.

In [29]:
from langchain_openai import OpenAIEmbeddings
 
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
 

# Stores the chunks in Chroma

In [ ]:
import os
 
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
 
# Check API key
api_key = os.getenv("OPENAI_API_KEY")
 
if not api_key:
    raise ValueError("OPENAI_API_KEY is missing in .env")
 
 
# Create OpenAI embedding model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
 
 
# Create vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
 
print("OpenAI embeddings created successfully!")
print("Chunks stored:", len(chunks))

OpenAI embeddings created successfully!
Chunks stored: 302


# Check for Similartity Search

In [32]:
query = "What is the main topic of this PDF?"
 
results = vectorstore.similarity_search(
    query,
    k=3
)
 
for i, result in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(result.page_content)


--- Result 1 ---
South & North Indian Recipes  |  Page 1
 SOUTH & NORTH INDIAN RECIPES
 A 200-page practical cookbook of traditional and everyday Indian dishes
 Recipe Collection
 
South Indian  North Indian  Regional Specialities  Snacks  Sweets  Cooking Tips

--- Result 2 ---
South & North Indian Recipes  |  Page 1
 SOUTH & NORTH INDIAN RECIPES
 A 200-page practical cookbook of traditional and everyday Indian dishes
 Recipe Collection
 
South Indian  North Indian  Regional Specialities  Snacks  Sweets  Cooking Tips

--- Result 3 ---
South & North Indian Recipes  |  Page 1
 SOUTH & NORTH INDIAN RECIPES
 A 200-page practical cookbook of traditional and everyday Indian dishes
 Recipe Collection
 
South Indian  North Indian  Regional Specialities  Snacks  Sweets  Cooking Tips


# Creates a retriever for searching the recipe content.

In [35]:
# -----------------------------------
# 1. Create retriever from Chroma
# -----------------------------------

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever created successfully!")


# -----------------------------------
# 2. Import required libraries
# -----------------------------------

from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_openai import ChatOpenAI


# -----------------------------------
# 3. Create LLM
# -----------------------------------

llm = ChatOpenAI(
    model="gpt-4o-mini"
)


# -----------------------------------
# 4. Create Retrieval QA chain
# -----------------------------------

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

print("RetrievalQA chain created successfully!")


# -----------------------------------
# 5. Ask question
# -----------------------------------

question = "What is the main topic of the document?"

response = qa.invoke({
    "query": question
})


# -----------------------------------
# 6. Print answer
# -----------------------------------

print("\nAnswer:")
print(response["result"])

Retriever created successfully!
RetrievalQA chain created successfully!

Answer:
The main topic of the document is South and North Indian recipes, including detailed cooking notes and serving ideas for various dishes.


# Create retriever from the existing Chroma vector store

In [36]:

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Retrieve the top 3 relevant chunks
top_chunks = retriever.invoke(question)

# Display the retrieved chunks
for i, doc in enumerate(top_chunks, start=1):
    print(f"\n{i}. {doc.page_content[:80]}")


1. South & North Indian Recipes  |  Page 30
Appam — Detailed Cooking Notes
Preparat

2. South & North Indian Recipes  |  Page 30
Appam — Detailed Cooking Notes
Preparat

3. South & North Indian Recipes  |  Page 30
Appam — Detailed Cooking Notes
Preparat
